### Knowledge Graph builder (deontic, party-centric)

#### 1. Setup — locate repo, load API keys

In [ ]:
import sys, os, json
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'server').exists():
    ROOT = ROOT.parent
SERVER = ROOT / 'server'
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

from dotenv import load_dotenv
load_dotenv(SERVER / '.env')

print('ROOT   :', ROOT)
print('OPENAI key set:', bool(os.getenv('OPENAI_API_KEY')))

#### 2. Config — provider, input/output folders

In [ ]:
PROVIDER = 'openai'   # only 'openai' (gpt-4.1) is wired; factory is extensible

PARAGRAPHS_DIR = ROOT / 'infra/json/paragraphs'
KG_OUT_DIR     = ROOT / 'infra/json/kg'
KG_OUT_DIR.mkdir(parents=True, exist_ok=True)

available = sorted(p.name for p in PARAGRAPHS_DIR.glob('*.json'))
for i, name in enumerate(available):
    print(i, name)

#### 3. Load one document's paragraphs

In [ ]:
DOC_INDEX = 0   # pick from the list above
DOC_FILE = available[DOC_INDEX]

src = json.load(open(PARAGRAPHS_DIR / DOC_FILE))
paragraphs = src['paragraphs']
doc_id = src['documentId']
print(doc_id)
print(len(paragraphs), 'paragraphs')

#### 4. Build the knowledge graph

Chunks the paragraphs, calls the LLM per chunk, and merges parties/clauses
across chunks (entity resolution). One LLM call per chunk — long contracts
take a bit.

In [ ]:
from services.graph.knowledge.extraction import build_knowledge_graph
from services.llm.factory import LLMProviderFactory

provider = LLMProviderFactory.create(PROVIDER)
kg = build_knowledge_graph(paragraphs, provider)

print('parties   :', len(kg.parties))
print('clauses   :', len(kg.clauses))
print('provisions:', len(kg.provisions))
print('edges     :', len(kg.edges))
print('types     :', dict(Counter(p.type for p in kg.provisions)))

#### 5. Save the KG

In [ ]:
out_path = KG_OUT_DIR / DOC_FILE
out_path.write_text(json.dumps(kg.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', out_path)

#### 6. Inspect — parties and sample provisions

In [ ]:
for p in kg.parties:
    print(f'[{p.id}] {p.name!r}  role={p.role!r}  aliases={p.aliases}')
print()
for pv in kg.provisions[:10]:
    print(f'[{pv.id}] {pv.type:11s} obligor={pv.obligorPartyId} benef={pv.beneficiaryPartyId} clause={pv.clauseId}')
    print('    ', pv.summary)

#### 7. (Optional) Batch — one KG per contract, for every file

Processes all documents in `infra/json/paragraphs/` at once. Each iteration
builds **one** knowledge graph for a whole contract (the internal chunking is
merged), and writes one KG file per contract to `infra/json/kg/`.

In [ ]:
for f in available:
    s = json.load(open(PARAGRAPHS_DIR / f))
    g = build_knowledge_graph(s['paragraphs'], provider)
    (KG_OUT_DIR / f).write_text(json.dumps(g.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{f[:55]:57s} provisions={len(g.provisions):4d} parties={len(g.parties)}')